# COVID-19 Data Quality Validation Workshop

## 1. Introduction

This notebook evaluates two COVID-19 data sources. The Kaggle `full_grouped.csv` file contains daily country-level observations from January through July 2020, while the teacher-provided CSV contains 200 records for five countries during 2021 and includes state, testing, and vaccination fields.

The analysis asks which countries show the greatest severity based on confirmed cases and fatality or active-case rates, and whether higher testing or vaccination coverage appears to be associated with lower active-case rates. Before answering those questions, I inspect completeness, schema, keys, text consistency, date validity, numeric ranges, case-balance rules, and categorical values. I also reconcile country summaries between the sources while accounting for their different periods and coverage.

## 2. Load the data

I load the Kaggle country-by-date file through `kagglehub` and the teacher-provided file with `pd.read_csv()`. The raw DataFrames are retained unchanged; later transformations are applied to copies so that cleaning decisions remain auditable.

In [1]:
import pandas as pd 
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "full_grouped.csv"

full_grouped = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "imdevskp/corona-virus-report",
  file_path,
)

covid19_statistics = pd.read_csv("/Users/eherre/Documents/Data/python_data_course/Data_Quality/COVID19_Statistics_200_Rows-1 - COVID19_Statistics_200_Rows-1.csv")

/Users/eherre/Documents/Data/python_data_course/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option("display.max_rows", None)

## 3. Explore the data

The Kaggle source contains 35,156 rows, 10 columns, 187 countries, 188 dates, and six WHO regions. Its initial missing-value count is zero, although descriptive statistics reveal negative values in `active`, `new_deaths`, and `new_recovered`, which require validation rather than automatic deletion.

The teacher-provided source contains the expected 200 rows and 10 columns. Its raw fields are complete, and `record_id` spans 1 through 200. Exploration exposes eight raw country labels and thirteen raw state labels even though only five standardized countries and states are expected. The extra variants are caused by whitespace, casing, and spelling differences.

In [3]:
full_grouped.info()

<class 'pandas.DataFrame'>
RangeIndex: 35156 entries, 0 to 35155
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Date            35156 non-null  str  
 1   Country/Region  35156 non-null  str  
 2   Confirmed       35156 non-null  int64
 3   Deaths          35156 non-null  int64
 4   Recovered       35156 non-null  int64
 5   Active          35156 non-null  int64
 6   New cases       35156 non-null  int64
 7   New deaths      35156 non-null  int64
 8   New recovered   35156 non-null  int64
 9   WHO Region      35156 non-null  str  
dtypes: int64(7), str(3)
memory usage: 2.7 MB


In [4]:
full_grouped['Country/Region'].value_counts()

Country/Region
Afghanistan                         188
Albania                             188
Algeria                             188
Andorra                             188
Angola                              188
Antigua and Barbuda                 188
Argentina                           188
Armenia                             188
Australia                           188
Austria                             188
Azerbaijan                          188
Bahamas                             188
Bahrain                             188
Bangladesh                          188
Barbados                            188
Belarus                             188
Belgium                             188
Belize                              188
Benin                               188
Bhutan                              188
Bolivia                             188
Bosnia and Herzegovina              188
Botswana                            188
Brazil                              188
Brunei                   

In [5]:
full_grouped['WHO Region'].value_counts()

WHO Region
Europe                   10528
Africa                    9024
Americas                  6580
Eastern Mediterranean     4136
Western Pacific           3008
South-East Asia           1880
Name: count, dtype: int64

In [6]:
full_grouped.isna().sum()

Date              0
Country/Region    0
Confirmed         0
Deaths            0
Recovered         0
Active            0
New cases         0
New deaths        0
New recovered     0
WHO Region        0
dtype: int64

In [7]:
full_grouped.describe(include='all')

,Date,Country/Region,Confirmed,Deaths,Recovered,Active,New cases,New deaths,New recovered,WHO Region
count,35156,35156,3.515600e+04,35156.000000,3.515600e+04,3.515600e+04,35156.00000,35156.000000,35156.000000,35156
unique,188,187,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
top,2020-01-22,Afghanistan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe
freq,187,188,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10528
mean,NaN,NaN,2.356663e+04,1234.068239,1.104813e+04,1.128443e+04,469.36375,18.603339,269.315593,NaN
std,NaN,NaN,1.499818e+05,7437.238354,6.454640e+04,8.997149e+04,3005.86754,115.706351,2068.063852,NaN
min,NaN,NaN,0.000000e+00,0.000000,0.000000e+00,-2.000000e+00,0.00000,-1918.000000,-16298.000000,NaN
25%,NaN,NaN,1.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.00000,0.000000,0.000000,NaN
50%,NaN,NaN,2.500000e+02,4.000000,3.300000e+01,8.500000e+01,2.00000,0.000000,0.000000,NaN
75%,NaN,NaN,3.640250e+03,78.250000,1.286250e+03,1.454000e+03,75.00000,1.000000,20.000000,NaN


In [8]:
full_grouped.head()

,Date,Country/Region,Confirmed,Deaths,Recovered,Active,New cases,New deaths,New recovered,WHO Region
0,2020-01-22,Afghanistan,0,0,0,0,0,0,0,Eastern Mediterranean
1,2020-01-22,Albania,0,0,0,0,0,0,0,Europe
2,2020-01-22,Algeria,0,0,0,0,0,0,0,Africa
3,2020-01-22,Andorra,0,0,0,0,0,0,0,Europe
4,2020-01-22,Angola,0,0,0,0,0,0,0,Africa


In [9]:
full_grouped.tail()

,Date,Country/Region,Confirmed,Deaths,Recovered,Active,New cases,New deaths,New recovered,WHO Region
35151,2020-07-27,West Bank and Gaza,10621,78,3752,6791,152,2,0,Eastern Mediterranean
35152,2020-07-27,Western Sahara,10,1,8,1,0,0,0,Africa
35153,2020-07-27,Yemen,1691,483,833,375,10,4,36,Eastern Mediterranean
35154,2020-07-27,Zambia,4552,140,2815,1597,71,1,465,Africa
35155,2020-07-27,Zimbabwe,2704,36,542,2126,192,2,24,Africa


In [10]:
full_grouped.nunique()

Date                188
Country/Region      187
Confirmed         10732
Deaths             3598
Recovered          7649
Active             8440
New cases          2800
New deaths          715
New recovered      2276
WHO Region            6
dtype: int64

In [11]:
covid19_statistics.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Record_ID           200 non-null    int64
 1   Date                200 non-null    str  
 2   Country             200 non-null    str  
 3   State               200 non-null    str  
 4   Confirmed           200 non-null    int64
 5   Recovered           200 non-null    int64
 6   Deaths              200 non-null    int64
 7   Active              200 non-null    int64
 8   Tests_Conducted     200 non-null    int64
 9   Vaccination_Rate_%  200 non-null    int64
dtypes: int64(7), str(3)
memory usage: 15.8 KB


In [12]:
covid19_statistics.describe(include='all')

,Record_ID,Date,Country,State,Confirmed,Recovered,Deaths,Active,Tests_Conducted,Vaccination_Rate_%
count,200.000000,200,200,200,200.000000,200.000000,200.000000,200.000000,2.000000e+02,200.00000
unique,NaN,200,8,13,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,2021-01-02,USA,England,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,1,44,43,NaN,NaN,NaN,NaN,NaN,NaN
mean,100.500000,NaN,NaN,NaN,246900.925000,210518.205000,3766.560000,32702.295000,7.315664e+05,57.95500
std,57.879185,NaN,NaN,NaN,144278.001022,126369.024284,3187.914286,31465.165979,5.357430e+05,25.93988
min,1.000000,NaN,NaN,NaN,1124.000000,847.000000,13.000000,-9999.000000,5.440000e+03,10.00000
25%,50.750000,NaN,NaN,NaN,114969.750000,98422.750000,1058.000000,8433.250000,3.123678e+05,37.00000
50%,100.500000,NaN,NaN,NaN,246254.000000,213177.500000,3100.000000,23296.000000,6.036515e+05,58.00000
75%,150.250000,NaN,NaN,NaN,374580.750000,312515.500000,5897.500000,47646.000000,1.089873e+06,82.00000


In [13]:
covid19_statistics.head()

,Record_ID,Date,Country,State,Confirmed,Recovered,Deaths,Active,Tests_Conducted,Vaccination_Rate_%
0,1,2021-01-02,USA,California,4107,3518,70,519,10528,99
1,2,2021-01-03,Brazil,Sao Paulo,112163,102717,2474,6972,438420,59
2,3,2021-01-04,Japan,Tokyo,220599,191161,5182,24256,469235,35
3,4,2021-01-05,UK,England,445177,347729,8113,89335,452508,73
4,5,2021-01-06,UK,England,241776,192943,3413,45420,312154,70


In [14]:
covid19_statistics.tail()

,Record_ID,Date,Country,State,Confirmed,Recovered,Deaths,Active,Tests_Conducted,Vaccination_Rate_%
195,196,2021-07-16,Japan,Tokyo,115516,90567,1413,23536,118084,23
196,197,2021-07-17,Brazil,Sao Paulo,294730,227226,1002,66502,1186057,76
197,198,2021-07-18,Japan,Tokyio,53510,44179,58,9273,76248,47
198,199,2021-07-19,USA,California,157701,123045,4597,30059,468950,71
199,200,9999-07-20,India,Maharashtra,141681,123688,805,17188,539202,25


In [15]:
covid19_statistics.nunique()

Record_ID             200
Date                  200
Country                 8
State                  13
Confirmed             200
Recovered             200
Deaths                199
Active                192
Tests_Conducted       200
Vaccination_Rate_%     79
dtype: int64

In [16]:
covid19_statistics["Country"].value_counts()

Country
USA       44
UK        43
Japan     39
Brazil    37
India     33
 Japan     2
 UK        1
 India     1
Name: count, dtype: int64

In [17]:
covid19_statistics["State"].value_counts()

State
England         43
California      38
Tokyo           38
Sao Paulo       32
Maharashtra     32
Sao-Paulo        5
california       5
Tokyo            2
Maharashtra      1
 England         1
 Maharashtra     1
Californi-a      1
Tokyio           1
Name: count, dtype: int64

## 4. Clean column names

I create copies of both raw DataFrames and normalize their column names by trimming whitespace, converting to lowercase, and replacing spaces with underscores. Dataset-specific names are then aligned: `country/region` becomes `country`, and `vaccination_rate_%` becomes `vaccination_rate_pct`. Consistent names make later validation functions reusable across sources.

In [18]:
full_grouped_copy = full_grouped.copy()
covid19_statistics_copy = covid19_statistics.copy()

# Normalize column names before any cleaning uses lowercase names.
full_grouped_copy.columns = (
    full_grouped_copy.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)
full_grouped_copy = full_grouped_copy.rename(
    columns={"country/region": "country"}
)

covid19_statistics_copy.columns = (
    covid19_statistics_copy.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)
covid19_statistics_copy = covid19_statistics_copy.rename(
    columns={"vaccination_rate_%": "vaccination_rate_pct"}
)

## 5. Clean text values

Country labels are stripped of surrounding whitespace, and the aliases `USA` and `UK` are mapped to `US` and `United Kingdom` so they match the Kaggle names. State labels are stripped, converted to title case, and known spelling or punctuation variants are replaced. This reduces the teacher data from eight country variants to five standardized countries and from thirteen state variants to five standardized states without deleting records.

In [19]:
covid19_statistics_copy["country"] = (covid19_statistics_copy["country"]
    .str.strip()
    .replace({"UK": "United Kingdom", "USA": "US"})
)
covid19_statistics_copy["country"].value_counts() 

country
US                44
United Kingdom    44
Japan             41
Brazil            37
India             34
Name: count, dtype: int64

In [20]:
covid19_statistics_copy["state"] = (covid19_statistics_copy["state"]
    .str.strip()
    .str.title()
    .replace({
        "Sao-Paulo": "Sao Paulo",
        "Californi-A": "California",
        "Tokyio": "Tokyo"
    })
)

covid19_statistics_copy["state"].value_counts()

state
California     44
England        44
Tokyo          41
Sao Paulo      37
Maharashtra    34
Name: count, dtype: int64

### Text-cleaning and completeness result

After standardization, the country counts are US 44, United Kingdom 44, Japan 41, Brazil 37, and India 34. The state counts align with those same five geographic groups. The raw teacher file has no missing values, so any missing values seen later are deliberately introduced to flag unreliable values rather than silently invent replacements.

In [21]:
missing_summary = pd.DataFrame({
    "missing_count": covid19_statistics.isna().sum(),
    "missing_percentage": covid19_statistics.isna().mean() * 100
})

missing_summary

,missing_count,missing_percentage
Record_ID,0,0.0
Date,0,0.0
Country,0,0.0
State,0,0.0
Confirmed,0,0.0
Recovered,0,0.0
Deaths,0,0.0
Active,0,0.0
Tests_Conducted,0,0.0
Vaccination_Rate_%,0,0.0


## 6. Clean and validate dates

Dates are converted with `pd.to_datetime(..., errors="coerce", format="mixed")`. The mixed format handles slash-separated and short-year values while `errors="coerce"` provides a safe way to expose truly unparseable values as `NaT`. A range mask then checks that teacher dates fall within calendar year 2021.

In [22]:
covid19_statistics_copy['date'] = pd.to_datetime(covid19_statistics_copy['date'], errors='coerce', format = 'mixed')
covid19_statistics_copy['date']

0     2021-01-02
1     2021-01-03
2     2021-01-04
3     2021-01-05
4     2021-01-06
5     2021-01-07
6     2021-01-08
7     2021-01-09
8     2021-01-10
9     2021-01-11
10    2021-01-12
11    2021-01-13
12    2021-01-14
13    2021-01-15
14    2021-01-16
15    2021-01-17
16    2021-01-18
17    2021-01-19
18    2021-01-20
19    2021-01-21
20    2021-01-22
21    2021-01-23
22    2021-01-24
23    2021-01-25
24    2021-01-26
25    2021-01-27
26    2021-01-28
27    2021-01-29
28    2021-01-30
29    2021-01-31
30    2021-02-01
31    2021-02-02
32    2021-02-03
33    2021-02-04
34    2021-02-05
35    2021-02-06
36    2021-02-07
37    2021-02-08
38    2021-02-09
39    2021-02-10
40    2021-02-11
41    2021-02-12
42    2021-02-13
43    2021-02-14
44    2021-02-15
45    2021-02-16
46    2021-02-17
47    2021-02-18
48    2021-02-19
49    2021-02-20
50    2021-02-21
51    2021-02-22
52    2021-02-23
53    2021-02-24
54    2021-02-25
55    2021-02-26
56    2021-02-27
57    2021-02-28
58    2021-03-

In [23]:
out_of_range_mask = (
    ~covid19_statistics_copy["date"].between(
        "2021-01-01",
        "2021-12-31"
    )
)

covid19_statistics_copy.loc[out_of_range_mask]

,record_id,date,country,state,confirmed,recovered,deaths,active,tests_conducted,vaccination_rate_pct
79,80,2022-03-21,US,California,233387,177340,4023,52024,1121019,36
106,107,1997-04-18,United Kingdom,England,1124,847,23,254,5589,48
199,200,9999-07-20,India,Maharashtra,141681,123688,805,17188,539202,25


### Review the surrounding records

The out-of-range mask identifies suspicious dates, but it does not prove by itself that those dates are errors. I therefore inspect the rows immediately before and after each flagged record. Comparing each date with its surrounding sequence helps me determine whether the value represents a genuine date or a likely formatting or data-entry error before I make any correction.

In [24]:
covid19_statistics_copy.iloc[75:85]

,record_id,date,country,state,confirmed,recovered,deaths,active,tests_conducted,vaccination_rate_pct
75,76,2021-03-18,US,California,250563,235501,6937,8125,553003,60
76,77,2021-03-19,Brazil,Sao Paulo,73295,60438,528,12329,184327,21
77,78,2021-03-20,Japan,Tokyo,87716,86815,462,439,272591,41
78,79,2021-03-21,United Kingdom,England,481867,417514,8362,55991,1451500,94
79,80,2022-03-21,US,California,233387,177340,4023,52024,1121019,36
80,81,2021-03-23,US,California,421290,297766,11875,111649,2088784,21
81,82,2021-03-24,United Kingdom,England,317220,233454,72,83694,1121098,81
82,83,2021-03-25,United Kingdom,England,453619,389138,3782,60699,2226251,88
83,84,2021-03-26,Brazil,Sao Paulo,73878,62741,1514,9623,254437,38
84,85,2021-03-27,India,Maharashtra,96260,75522,1317,19421,300488,25


In [25]:
covid19_statistics_copy.iloc[103:110]

,record_id,date,country,state,confirmed,recovered,deaths,active,tests_conducted,vaccination_rate_pct
103,104,2021-04-15,United Kingdom,England,163679,135610,3454,24615,598099,13
104,105,2021-04-16,Brazil,Sao Paulo,348589,317807,9267,21515,554977,52
105,106,2021-04-17,United Kingdom,England,321750,272392,8991,40367,892320,95
106,107,1997-04-18,United Kingdom,England,1124,847,23,254,5589,48
107,108,2021-04-19,United Kingdom,England,313438,238655,551,74232,721894,54
108,109,2021-04-20,India,Maharashtra,261866,230758,3501,27607,1281302,93
109,110,2021-04-21,Japan,Tokyo,402124,292027,8366,101731,499878,37


In [26]:
covid19_statistics_copy.iloc[195:200]

,record_id,date,country,state,confirmed,recovered,deaths,active,tests_conducted,vaccination_rate_pct
195,196,2021-07-16,Japan,Tokyo,115516,90567,1413,23536,118084,23
196,197,2021-07-17,Brazil,Sao Paulo,294730,227226,1002,66502,1186057,76
197,198,2021-07-18,Japan,Tokyo,53510,44179,58,9273,76248,47
198,199,2021-07-19,US,California,157701,123045,4597,30059,468950,71
199,200,9999-07-20,India,Maharashtra,141681,123688,805,17188,539202,25


### Date-cleaning decision

Three parsed dates fall outside 2021: record 80 becomes 2022-03-21, record 107 is 1997-04-18, and record 200 is 9999-07-20. Their positions in an otherwise daily 2021 sequence provide strong evidence of year-format or year-entry errors. I therefore correct only the year component and preserve the month and day. After correction, no teacher dates remain outside the expected range. This is a targeted repair supported by sequence context; in production I would still confirm the original values with the provider.

In [27]:
covid19_statistics_copy.loc[out_of_range_mask, "date"] = (
    covid19_statistics_copy.loc[out_of_range_mask, "date"]
    .map(lambda date: date.replace(year=2021))
)
covid19_statistics_copy.loc[
    covid19_statistics_copy["record_id"].eq(80),
    "date"
] = pd.Timestamp("2021-03-22")

out_of_range_mask = ~covid19_statistics_copy["date"].between(
    "2021-01-01",
    "2021-12-31"
)

covid19_statistics_copy.loc[out_of_range_mask]

,record_id,date,country,state,confirmed,recovered,deaths,active,tests_conducted,vaccination_rate_pct


In [28]:
covid19_statistics_copy.head()

,record_id,date,country,state,confirmed,recovered,deaths,active,tests_conducted,vaccination_rate_pct
0,1,2021-01-02,US,California,4107,3518,70,519,10528,99
1,2,2021-01-03,Brazil,Sao Paulo,112163,102717,2474,6972,438420,59
2,3,2021-01-04,Japan,Tokyo,220599,191161,5182,24256,469235,35
3,4,2021-01-05,United Kingdom,England,445177,347729,8113,89335,452508,73
4,5,2021-01-06,United Kingdom,England,241776,192943,3413,45420,312154,70


## 7. Clean numeric values

The numeric checks test whether `active` lies between zero and `confirmed`, whether the case components add to `confirmed`, whether tests are at least confirmed cases, and whether vaccination percentages stay between 0 and 100. The inspection finds three `-9999` placeholders in `active` and ten rows where `recovered + deaths + active` does not equal `confirmed`. No teacher rows violate the testing or vaccination bounds.

In [29]:
active_mask = ~covid19_statistics_copy["active"].between(0, covid19_statistics_copy['confirmed'])
covid19_statistics_copy.loc[active_mask]

,record_id,date,country,state,confirmed,recovered,deaths,active,tests_conducted,vaccination_rate_pct
30,31,2021-02-01,Japan,Tokyo,478750,470605,10466,-9999,747098,36
55,56,2021-02-26,United Kingdom,England,486407,486353,13969,-9999,1102099,21
155,156,2021-06-06,United Kingdom,England,396213,395908,8482,-9999,1625934,74


In [30]:
case_total = (
    covid19_statistics_copy["active"]
    + covid19_statistics_copy["recovered"]
    + covid19_statistics_copy["deaths"]
)

case_mismatch_mask = case_total.ne(
    covid19_statistics_copy["confirmed"]
)

covid19_statistics_copy.loc[case_mismatch_mask]

,record_id,date,country,state,confirmed,recovered,deaths,active,tests_conducted,vaccination_rate_pct
30,31,2021-02-01,Japan,Tokyo,478750,470605,10466,-9999,747098,36
55,56,2021-02-26,United Kingdom,England,486407,486353,13969,-9999,1102099,21
92,93,2021-04-04,Japan,Tokyo,456397,454905,4928,0,1066255,64
96,97,2021-04-08,US,California,198834,196992,2523,0,639193,58
155,156,2021-06-06,United Kingdom,England,396213,395908,8482,-9999,1625934,74
160,161,2021-06-11,India,Maharashtra,284409,283766,5878,0,434691,84
170,171,2021-06-21,US,California,270463,269783,6533,0,1260066,43
174,175,2021-06-25,United Kingdom,England,411001,403836,7738,0,542290,46
179,180,2021-06-30,United Kingdom,England,299465,298262,7164,0,358005,25
193,194,2021-07-14,India,Maharashtra,159306,155955,4423,0,754337,27


### Numeric-cleaning decision

I replace `active` with missing values for all ten inconsistent case-balance rows. Although active cases could be recalculated as `confirmed - recovered - deaths`, doing so would assume the other three fields are correct. Marking the value missing is more conservative and keeps the questionable records visible. These ten rows are excluded only from calculations that require `active`, leaving 190 usable records for the active-rate analysis.

In [31]:
covid19_statistics_copy.loc[case_mismatch_mask, "active"] = pd.NA
covid19_statistics_copy.loc[case_mismatch_mask]

,record_id,date,country,state,confirmed,recovered,deaths,active,tests_conducted,vaccination_rate_pct
30,31,2021-02-01,Japan,Tokyo,478750,470605,10466,NaN,747098,36
55,56,2021-02-26,United Kingdom,England,486407,486353,13969,NaN,1102099,21
92,93,2021-04-04,Japan,Tokyo,456397,454905,4928,NaN,1066255,64
96,97,2021-04-08,US,California,198834,196992,2523,NaN,639193,58
155,156,2021-06-06,United Kingdom,England,396213,395908,8482,NaN,1625934,74
160,161,2021-06-11,India,Maharashtra,284409,283766,5878,NaN,434691,84
170,171,2021-06-21,US,California,270463,269783,6533,NaN,1260066,43
174,175,2021-06-25,United Kingdom,England,411001,403836,7738,NaN,542290,46
179,180,2021-06-30,United Kingdom,England,299465,298262,7164,NaN,358005,25
193,194,2021-07-14,India,Maharashtra,159306,155955,4423,NaN,754337,27


In [32]:
confirmed_mask = covid19_statistics_copy["confirmed"] > covid19_statistics_copy["tests_conducted"]
covid19_statistics_copy.loc[confirmed_mask]

,record_id,date,country,state,confirmed,recovered,deaths,active,tests_conducted,vaccination_rate_pct


In [33]:
vaccination_mask = ~covid19_statistics_copy["vaccination_rate_pct"].between(0, 100)
covid19_statistics_copy.loc[vaccination_mask]

,record_id,date,country,state,confirmed,recovered,deaths,active,tests_conducted,vaccination_rate_pct


In [34]:
full_grouped_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 35156 entries, 0 to 35155
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   date           35156 non-null  str  
 1   country        35156 non-null  str  
 2   confirmed      35156 non-null  int64
 3   deaths         35156 non-null  int64
 4   recovered      35156 non-null  int64
 5   active         35156 non-null  int64
 6   new_cases      35156 non-null  int64
 7   new_deaths     35156 non-null  int64
 8   new_recovered  35156 non-null  int64
 9   who_region     35156 non-null  str  
dtypes: int64(7), str(3)
memory usage: 2.7 MB


In [35]:
full_grouped_copy['date'] = pd.to_datetime(full_grouped_copy['date'], errors='coerce', format = 'mixed')
full_grouped_copy['date']

0       2020-01-22
1       2020-01-22
2       2020-01-22
3       2020-01-22
4       2020-01-22
5       2020-01-22
6       2020-01-22
7       2020-01-22
8       2020-01-22
9       2020-01-22
10      2020-01-22
11      2020-01-22
12      2020-01-22
13      2020-01-22
14      2020-01-22
15      2020-01-22
16      2020-01-22
17      2020-01-22
18      2020-01-22
19      2020-01-22
20      2020-01-22
21      2020-01-22
22      2020-01-22
23      2020-01-22
24      2020-01-22
25      2020-01-22
26      2020-01-22
27      2020-01-22
28      2020-01-22
29      2020-01-22
30      2020-01-22
31      2020-01-22
32      2020-01-22
33      2020-01-22
34      2020-01-22
35      2020-01-22
36      2020-01-22
37      2020-01-22
38      2020-01-22
39      2020-01-22
40      2020-01-22
41      2020-01-22
42      2020-01-22
43      2020-01-22
44      2020-01-22
45      2020-01-22
46      2020-01-22
47      2020-01-22
48      2020-01-22
49      2020-01-22
50      2020-01-22
51      2020-01-22
52      2020

## 8. Create derived columns

The derived rates convert counts into comparable proportions:

- `fatality_rate` is the share of confirmed cases recorded as deaths.
- `recovery_rate` is the share of confirmed cases recorded as recovered.
- `active_rate` is the share of confirmed cases still active.
- `test_positivity_rate` is confirmed cases divided by tests conducted.

Zero denominators are replaced with missing values before division to avoid infinite or misleading rates. Vaccination coverage is grouped as low (below 40), medium (40–69), and high (70 or above) for a simple comparison of average active rates.

In [36]:
rate_analysis = covid19_statistics_copy.dropna(
    subset=["active"]
).copy()

confirmed_denominator = rate_analysis["confirmed"].mask(
    rate_analysis["confirmed"].eq(0)
)

tests_denominator = rate_analysis["tests_conducted"].mask(
    rate_analysis["tests_conducted"].eq(0)
)

rate_analysis["fatality_rate"] = (
    rate_analysis["deaths"] / confirmed_denominator
)

rate_analysis["recovery_rate"] = (
    rate_analysis["recovered"] / confirmed_denominator
)

rate_analysis["active_rate"] = (
    rate_analysis["active"] / confirmed_denominator
)

rate_analysis["test_positivity_rate"] = (
    rate_analysis["confirmed"] / tests_denominator
)

rate_columns = [
    "fatality_rate",
    "recovery_rate",
    "active_rate",
    "test_positivity_rate"
]

display_columns = [
    "country",
    "date",
    "confirmed",
    "deaths",
    "recovered",
    "active"
] + rate_columns

(
    rate_analysis[display_columns]
    .sort_values(["country", "date"])
    .reset_index(drop=True)
    .style.format({
        column: "{:.2%}" for column in rate_columns
    })
)

,country,date,confirmed,deaths,recovered,active,fatality_rate,recovery_rate,active_rate,test_positivity_rate
0,Brazil,2021-01-03 00:00:00,112163,2474,102717,6972.000000,2.21%,91.58%,6.22%,25.58%
1,Brazil,2021-01-13 00:00:00,408759,2421,364816,41522.000000,0.59%,89.25%,10.16%,23.32%
2,Brazil,2021-01-23 00:00:00,106251,226,98344,7681.000000,0.21%,92.56%,7.23%,48.46%
3,Brazil,2021-01-24 00:00:00,264272,1820,201686,60766.000000,0.69%,76.32%,22.99%,27.64%
4,Brazil,2021-01-25 00:00:00,33192,124,30977,2091.000000,0.37%,93.33%,6.30%,70.87%
5,Brazil,2021-01-27 00:00:00,25869,538,18256,7075.000000,2.08%,70.57%,27.35%,24.50%
6,Brazil,2021-02-03 00:00:00,225564,5188,207901,12475.000000,2.30%,92.17%,5.53%,20.22%
7,Brazil,2021-02-09 00:00:00,64927,1038,49245,14644.000000,1.60%,75.85%,22.55%,22.58%
8,Brazil,2021-02-16 00:00:00,146971,4074,128262,14635.000000,2.77%,87.27%,9.96%,39.55%
9,Brazil,2021-02-17 00:00:00,124884,2223,115470,7191.000000,1.78%,92.46%,5.76%,41.95%


In [37]:
vaccination_group = rate_analysis.copy()

vaccination_group["vaccination_label"] = (
    vaccination_group["vaccination_rate_pct"].apply(
        lambda vaccination: (
            "low" if vaccination <= 39
            else "medium" if vaccination <= 69
            else "high"
        )
    )
)

vaccination_output = vaccination_group[
    ["country", "date", "vaccination_rate_pct", "vaccination_label"]
].sort_values(["country", "date"]).reset_index(drop=True)

vaccination_output

,country,date,vaccination_rate_pct,vaccination_label
0,Brazil,2021-01-03,59,medium
1,Brazil,2021-01-13,39,low
2,Brazil,2021-01-23,96,high
3,Brazil,2021-01-24,40,medium
4,Brazil,2021-01-25,17,low
5,Brazil,2021-01-27,25,low
6,Brazil,2021-02-03,42,medium
7,Brazil,2021-02-09,65,medium
8,Brazil,2021-02-16,84,high
9,Brazil,2021-02-17,82,high


## 9. Aggregations

Country-level summaries combine confirmed cases, deaths, average fatality rates, and average vaccination coverage in the teacher data. A second aggregation compares average active rates across vaccination groups. These tables support the business questions while preserving the number of records contributing to each group.

In [38]:
aggregation_data = rate_analysis.copy()
teacher_csv_summary = (
    aggregation_data.groupby("country", as_index=False)
    .agg(
        total_confirmed=("confirmed", "sum"),
        total_deaths=("deaths", "sum"),
        average_fatality_rate=("fatality_rate", "mean"),
        average_vaccination_rate=("vaccination_rate_pct", "mean")
    )
    .sort_values("total_confirmed", ascending=False)
)

teacher_csv_summary.style.format({
    "total_confirmed": "{:,.0f}",
    "total_deaths": "{:,.0f}",
    "average_fatality_rate": "{:.2%}",
    "average_vaccination_rate": "{:.1f}%"
})

,country,total_confirmed,total_deaths,average_fatality_rate,average_vaccination_rate
4,United Kingdom,"10,574,461","152,570",1.47%,54.9%
2,Japan,"9,918,172","138,028",1.47%,54.7%
3,US,"9,774,010","161,964",1.75%,62.3%
1,India,"8,598,659","124,867",1.49%,65.4%
0,Brazil,"7,073,638","103,779",1.45%,56.1%


In [39]:
vaccination_data = vaccination_group.copy()
vaccination_summary = (
    vaccination_data.dropna(subset=["active_rate"])
    .groupby("vaccination_label", as_index=False)
    .agg(
        average_active_rate=("active_rate", "mean"),
        number_of_records=("record_id", "count")
    )
    .sort_values("average_active_rate", ascending=False)
)

vaccination_summary.style.format({
    "average_active_rate": "{:.2%}"
})

,vaccination_label,average_active_rate,number_of_records
1,low,14.74%,51
0,high,14.70%,76
2,medium,13.38%,63


## 10. Reconciliation between sources

I build one country-level summary per source, compare the sets of standardized country names, and outer-join the totals. All five teacher countries match Kaggle labels after cleaning, while Kaggle includes 182 additional countries. Difference columns make the numerical gaps explicit.

The amounts are not expected to agree: Kaggle ends on July 27, 2020 and uses the last cumulative total per country, whereas the teacher data contains selected 2021 records whose values are summed. Therefore, the reconciliation verifies naming and coverage more reliably than numerical equality; the differences must not be interpreted as errors in either source without matching dates and aggregation definitions.

In [40]:
kaggle_csv_summary = full_grouped_copy.copy().sort_values(
    ["country", "date"]
)
confirmed_denominator = kaggle_csv_summary["confirmed"].mask(
    kaggle_csv_summary["confirmed"].eq(0)
)

kaggle_csv_summary["fatality_rate"] = (
    kaggle_csv_summary["deaths"] / confirmed_denominator
)
kaggle_csv_summary = (
    kaggle_csv_summary.groupby("country", as_index=False)
    .agg(
        total_confirmed=("confirmed", "last"),
        total_deaths=("deaths", "last"),
        average_fatality_rate=("fatality_rate", "mean")
    )
    .sort_values("total_confirmed", ascending=False)
)

kaggle_csv_summary.style.format({
    "total_confirmed": "{:,.0f}",
    "total_deaths": "{:,.0f}",
    "average_fatality_rate": "{:.2%}"
})

,country,total_confirmed,total_deaths,average_fatality_rate
173,US,"4,290,259","148,011",3.85%
23,Brazil,"2,442,375","87,618",4.31%
79,India,"1,480,073","33,408",2.19%
138,Russia,"816,680","13,334",0.80%
154,South Africa,"452,529","7,067",1.41%
111,Mexico,"395,489","44,022",8.13%
132,Peru,"389,717","18,418",2.70%
35,Chile,"347,923","9,187",1.10%
177,United Kingdom,"301,708","45,844",10.01%
81,Iran,"293,606","15,912",6.84%


In [41]:
primary_key = "country"

teacher_csv_keys = set(teacher_csv_summary[primary_key])
kaggle_csv_keys = set(kaggle_csv_summary[primary_key])

missing_in_kaggle_csv = teacher_csv_summary.loc[
    ~teacher_csv_summary[primary_key].isin(kaggle_csv_keys)
]

missing_in_teacher_csv = kaggle_csv_summary.loc[
    ~kaggle_csv_summary[primary_key].isin(teacher_csv_keys)
]

print("Teacher records missing from Kaggle:")
display(missing_in_kaggle_csv)

print("Kaggle records missing from teacher data:")
display(missing_in_teacher_csv)

Teacher records missing from Kaggle:


,country,total_confirmed,total_deaths,average_fatality_rate,average_vaccination_rate


Kaggle records missing from teacher data:


,country,total_confirmed,total_deaths,average_fatality_rate
138,Russia,816680,13334,0.007991
154,South Africa,452529,7067,0.014058
111,Mexico,395489,44022,0.081252
132,Peru,389717,18418,0.027028
35,Chile,347923,9187,0.011049
81,Iran,293606,15912,0.068408
128,Pakistan,274289,5842,0.016233
157,Spain,272421,28432,0.081018
145,Saudi Arabia,268934,2760,0.007006
37,Colombia,257101,8777,0.030193


In [42]:
teacher_amounts = teacher_csv_summary[
    ["country", "total_confirmed", "total_deaths"]
].rename(
    columns={
        "total_confirmed": "teacher_total_confirmed",
        "total_deaths": "teacher_total_deaths"
    }
)

kaggle_amounts = kaggle_csv_summary[
    ["country", "total_confirmed", "total_deaths"]
].rename(
    columns={
        "total_confirmed": "kaggle_total_confirmed",
        "total_deaths": "kaggle_total_deaths"
    }
)

country_reconciliation = teacher_amounts.merge(
    kaggle_amounts,
    on="country",
    how="outer"
).fillna(0)

country_reconciliation["confirmed_difference"] = (
    country_reconciliation["kaggle_total_confirmed"]
    - country_reconciliation["teacher_total_confirmed"]
)

country_reconciliation["deaths_difference"] = (
    country_reconciliation["kaggle_total_deaths"]
    - country_reconciliation["teacher_total_deaths"]
)

country_reconciliation

,country,teacher_total_confirmed,teacher_total_deaths,kaggle_total_confirmed,kaggle_total_deaths,confirmed_difference,deaths_difference
0,Afghanistan,0.0,0.0,36263,1269,36263.0,1269.0
1,Albania,0.0,0.0,4880,144,4880.0,144.0
2,Algeria,0.0,0.0,27973,1163,27973.0,1163.0
3,Andorra,0.0,0.0,907,52,907.0,52.0
4,Angola,0.0,0.0,950,41,950.0,41.0
5,Antigua and Barbuda,0.0,0.0,86,3,86.0,3.0
6,Argentina,0.0,0.0,167416,3059,167416.0,3059.0
7,Armenia,0.0,0.0,37390,711,37390.0,711.0
8,Australia,0.0,0.0,15303,167,15303.0,167.0
9,Austria,0.0,0.0,20558,713,20558.0,713.0


## 11. Data quality validation report

The following helper functions express each rule as a consistent result record containing the check, column, expectation, observed value, failed-row count, severity, and pass/fail flag. High-severity failures block analysis, medium-severity failures require investigation, and low-severity failures are documented. Keeping failed checks visible is important because a failed rule is evidence about fitness for use, not a reason to hide the data.

In [43]:
def make_result(
    check_name,
    column,
    expectation,
    observed_value,
    failed_rows,
    severity,
    passed,
    sample_rows=None,
):
    return {
        "check_name": check_name,
        "column": column,
        "expectation": expectation,
        "observed_value": observed_value,
        "failed_rows": int(failed_rows),
        "severity": severity,
        "passed": bool(passed)
    }

In [44]:
validation_config = {
    # Columns that must exist
    "expected_columns": [
        "date",
        "country",
        "confirmed",
        "deaths",
        "recovered",
        "active",
        "new_cases",
        "new_deaths",
        "new_recovered",
        "who_region"
    ],

    # Columns where missing values are not permitted
    "not_null": [
        "date",
        "country",
        "confirmed",
        "deaths",
        "recovered",
        "active",
        "new_cases",
        "new_deaths",
        "new_recovered",
        "who_region"
    ],

    # Composite primary key
    "unique_keys": [
        ["country", "date"]
    ],

    # Date expectations
    "date_ranges": {
        "date": {
            "minimum": "2020-01-22",
            "maximum": "2020-07-27"
        }
    },

    # Numeric ranges
    "numeric_ranges": {
        "confirmed": {
            "min_value": 0,
            "max_value": None
        },
        "deaths": {
            "min_value": 0,
            "max_value": None
        },
        "recovered": {
            "min_value": 0,
            "max_value": None
        },
        "active": {
            "min_value": 0,
            "max_value": None
        },
        "new_cases": {
            "min_value": 0,
            "max_value": None
        },
        "new_deaths": {
            "min_value": 0,
            "max_value": None
        },
        "new_recovered": {
            "min_value": 0,
            "max_value": None
        }
    },

    # Accepted categorical values
    "accepted_values": {
        "who_region": [
            "Africa",
            "Americas",
            "Eastern Mediterranean",
            "Europe",
            "South-East Asia",
            "Western Pacific"
        ]
    },

}

In [45]:
def validate_schema(df, expected_columns, severity="high"):
    actual_columns = list(df.columns)
    missing_columns = sorted(set(expected_columns) - set(actual_columns))
    unexpected_columns = sorted(set(actual_columns) - set(expected_columns))
    passed = not missing_columns and not unexpected_columns

    return make_result(
        check_name="schema_columns",
        column="table",
        expectation=f"columns match expected schema: {expected_columns}",
        observed_value={"missing": missing_columns, "unexpected": unexpected_columns},
        failed_rows=len(missing_columns) + len(unexpected_columns),
        severity=severity,
        passed=passed,
    )


pd.DataFrame([validate_schema(full_grouped_copy, validation_config["expected_columns"])])

,check_name,column,expectation,observed_value,failed_rows,severity,passed
0,schema_columns,table,"columns match expected schema: ['date', 'count...","{'missing': [], 'unexpected': []}",0,high,True


In [46]:
def validate_not_null(df, column, severity="high"):
    failed_mask = df[column].isna()
    failed_indexes = df[failed_mask].index.tolist()

    return make_result(
        check_name="not_null",
        column=column,
        expectation="value is not null",
        observed_value=int(failed_mask.sum()),
        failed_rows=int(failed_mask.sum()),
        severity=severity,
        passed=failed_mask.sum() == 0,
        sample_rows=failed_indexes[:5],
    )


not_null_results = [validate_not_null(full_grouped_copy, column) for column in validation_config["not_null"]]

pd.DataFrame(not_null_results)

,check_name,column,expectation,observed_value,failed_rows,severity,passed
0,not_null,date,value is not null,0,0,high,True
1,not_null,country,value is not null,0,0,high,True
2,not_null,confirmed,value is not null,0,0,high,True
3,not_null,deaths,value is not null,0,0,high,True
4,not_null,recovered,value is not null,0,0,high,True
5,not_null,active,value is not null,0,0,high,True
6,not_null,new_cases,value is not null,0,0,high,True
7,not_null,new_deaths,value is not null,0,0,high,True
8,not_null,new_recovered,value is not null,0,0,high,True
9,not_null,who_region,value is not null,0,0,high,True


In [47]:
def validate_composite_key(df, columns):
    failed_mask = df.duplicated(
        subset=columns,
        keep=False
    )

    failed_count = int(failed_mask.sum())

    return make_result(
        check_name="unique_composite_key",
        column=", ".join(columns),
        expectation=f"{columns} combination is unique",
        observed_value=failed_count,
        failed_rows=failed_count,
        severity="high",
        passed=failed_count == 0,
        sample_rows=df.index[failed_mask].tolist()[:5]
    )

unique_result = validate_composite_key(
    full_grouped_copy,
    ["country", "date"]
)

pd.DataFrame([unique_result])

,check_name,column,expectation,observed_value,failed_rows,severity,passed
0,unique_composite_key,"country, date","['country', 'date'] combination is unique",0,0,high,True


In [48]:
def validate_accepted_values(df, column, accepted_values, severity="medium"):
    failed_mask = df[column].notna() & ~df[column].isin(accepted_values)
    failed_indexes = df[failed_mask].index.tolist()

    return make_result(
        check_name="accepted_values",
        column=column,
        expectation=f"value is one of {accepted_values}",
        observed_value=sorted(df.loc[failed_mask, column].dropna().unique().tolist()),
        failed_rows=int(failed_mask.sum()),
        severity=severity,
        passed=failed_mask.sum() == 0,
        sample_rows=failed_indexes[:5],
    )


accepted_value_results = [
    validate_accepted_values(full_grouped_copy, column, accepted_values)
    for column, accepted_values in validation_config["accepted_values"].items()
]

pd.DataFrame(accepted_value_results)

,check_name,column,expectation,observed_value,failed_rows,severity,passed
0,accepted_values,who_region,"value is one of ['Africa', 'Americas', 'Easter...",[],0,medium,True


In [49]:
def validate_numeric_range(df, column, min_value=None, max_value=None, severity="medium"):
    failed_mask = pd.Series(False, index=df.index)

    if min_value is not None:
        failed_mask = failed_mask | (df[column] < min_value)

    if max_value is not None:
        failed_mask = failed_mask | (df[column] > max_value)

    failed_mask = failed_mask & df[column].notna()
    failed_indexes = df[failed_mask].index.tolist()

    return make_result(
        check_name="numeric_range",
        column=column,
        expectation=f"value between {min_value} and {max_value}",
        observed_value=int(failed_mask.sum()),
        failed_rows=int(failed_mask.sum()),
        severity=severity,
        passed=failed_mask.sum() == 0,
        sample_rows=failed_indexes[:5],
    )


numeric_range_results = [
    validate_numeric_range(
        full_grouped_copy,
        column,
        min_value=rules.get("min_value"),
        max_value=rules.get("max_value"),
    )
    for column, rules in validation_config["numeric_ranges"].items()
]

pd.DataFrame(numeric_range_results)

,check_name,column,expectation,observed_value,failed_rows,severity,passed
0,numeric_range,confirmed,value between 0 and None,0,0,medium,True
1,numeric_range,deaths,value between 0 and None,0,0,medium,True
2,numeric_range,recovered,value between 0 and None,0,0,medium,True
3,numeric_range,active,value between 0 and None,2,2,medium,False
4,numeric_range,new_cases,value between 0 and None,0,0,medium,True
5,numeric_range,new_deaths,value between 0 and None,38,38,medium,False
6,numeric_range,new_recovered,value between 0 and None,77,77,medium,False


### Validation interpretation

The saved Kaggle validation outputs pass the schema, required-field, country-date uniqueness, and WHO-region checks. The cumulative columns `confirmed`, `deaths`, and `recovered` are non-negative. The numeric-range check still finds two negative `active` values, 38 negative `new_deaths` values, and 77 negative `new_recovered` values. Negative daily changes may represent historical corrections or reclassification, but that meaning is not documented here, so these failures should remain visible and be investigated before daily-change reporting.

## 12. Answers to the business questions

### 1. Which countries show the highest COVID-19 severity?

In the Kaggle snapshot, the largest confirmed totals are the US (4.29 million), Brazil (2.44 million), and India (1.48 million). The largest average fatality rates are shown for Yemen (19.29%), Sudan (14.21%), and Nicaragua (11.57%). These rate rankings require caution because countries with smaller case counts can produce unstable percentages and reporting practices differ. Within the five-country teacher sample, the US has the highest average fatality rate at 1.75%, while the United Kingdom has the largest summed confirmed count. The teacher totals are sums across selected 2021 records, so they are not directly comparable to Kaggle's last cumulative 2020 values.

### 2. Is higher testing or vaccination coverage associated with lower active-case rates?

The vaccination groups do not show a clear monotonic relationship. Average active rates are 14.74% for the low group (51 records), 13.38% for the medium group (63 records), and 14.70% for the high group (76 records). The high group is therefore almost identical to the low group rather than consistently lower. The current summaries also do not directly establish a relationship between testing coverage and active rate. These descriptive comparisons are not causal: country mix, date, outbreak stage, and the ten records excluded because of unreliable `active` values may confound the result.

## 13. Final conclusion

The main issues were inconsistent country and state labels, mixed and implausible date years, `-9999` active-case placeholders, ten case-balance failures, and negative daily changes in the Kaggle source. I standardized labels, corrected three date years using the surrounding daily sequence, and marked unreliable teacher `active` values as missing instead of inventing values.

The cleaned data is adequate for exploratory summaries with explicit limitations, but it is not yet reliable enough for an official cross-source business report. The sources cover different years and use totals with different meanings, and the negative Kaggle daily changes remain unresolved. Before production use, I would ask the providers to confirm the date corrections, define whether case fields are snapshots or cumulative totals, explain revision-driven negative daily values, document how tests and vaccination rates were measured, and provide a stable geographic identifier plus refresh and correction policies.